In [773]:
import behaviors
import no_signaling_sets
import numpy as np
import samplers
import scipy
from loguru import logger


In [774]:
verbose = False

delta = 2
m = 2

In [775]:
sampler = samplers.NoSignalingSampler(delta, m)

sampled_behavior = sampler.sample()

srns_set = no_signaling_sets.ShortRangeNoSignalingSet(
    delta=delta,
    m=m,
    measured_behavior=sampled_behavior,
)

A_eq, b_eq = srns_set.get_equations(measured_behavior=sampled_behavior)

q_shape = behaviors.LatentSRNSBehavior(
    delta=delta,
    m=m,
).vector_shape[0]

lb = np.zeros(q_shape+1)
rb = np.ones(q_shape+1)
rb[0] = 2

bounds = [(lb[i], rb[i]) for i in range(q_shape+1)]

c = np.zeros(q_shape+1)
c[0] = -1

if verbose:
    with np.printoptions(threshold=np.inf):
        print("A_eq")
        print(A_eq)
        print("b_eq")
        print(b_eq)
        print("bounds")
        print(bounds)
        print("c")
        print(c)
        print('\n\n---------\n\n')

print(f"Sampled behavior : {sampled_behavior}")
print(f"Sampled behavior is tested [{sampled_behavior.is_no_signaling()}] to being no signaling")


2025-05-19 16:46:20.773 | SUCCESS  | samplers:sample_multiple:100 - Samples shape: (1, 32)
2025-05-19 16:46:20.774 | DEBUG    | no_signaling_sets:express_as_function_of_q:189 - Estimated memory complexity of lacking_betas: 56 bytes


Sampled behavior : Behavior:
Short path (z=S):
[[0.14002255 0.45017597 0.16821806 0.22374415]
 [0.47452641 0.164373   0.08772366 0.03219757]
 [0.29423354 0.085559   0.26603804 0.31199081]
 [0.09121749 0.29989204 0.47802024 0.43206746]]
Long path (z=L) :
[[0.59578366 0.03575622 0.13876188 0.04672246]
 [0.0187653  0.57879275 0.11717985 0.20921926]
 [0.27190581 0.25935572 0.72892759 0.24838947]
 [0.11354523 0.12609531 0.01513068 0.49566881]]
------------
Sampled behavior is tested [True] to being no signaling


In [776]:
import scipy.optimize

res:scipy.optimize.OptimizeResult = scipy.optimize.linprog(
    c=c,
    A_eq=A_eq,
    b_eq=b_eq,
    bounds=bounds,
)

print(res.success)
print(-res.fun)
print(res.status)
print(res.x)

True
1.0644217103001925
0
[1.06442171 0.13293762 0.46307165 0.16294953 0.22205271 0.48899079
 0.15885676 0.07726955 0.01816637 0.29708314 0.07496543 0.26707123
 0.31598437 0.08098845 0.30310617 0.4927097  0.44379656 0.02195427
 0.         0.59610537 0.13159573 0.         0.03362698 0.00386877
 0.07499637 0.25995843 0.24828572 0.01335859 0.51149521 0.
 0.         0.10475458 0.        ]


In [777]:
yielded_behavior = behaviors.LatentSRNSBehavior(
    delta=delta,
    m=m,
    vector=np.clip(np.array(res.x[1:]), 0, 1),
)
print(f"Yielded behavior is {yielded_behavior}")
print(f"Yielded behavior is tested [{yielded_behavior.is_no_signaling()}] to being no signaling")
print(f"Yielded behavior is tested [{yielded_behavior.is_normalized()}] to being normalized")


Yielded behavior is Behavior:
Short path (z=S):
[[0.13293762 0.46307165 0.16294953 0.22205271]
 [0.48899079 0.15885676 0.07726955 0.01816637]
 [0.29708314 0.07496543 0.26707123 0.31598437]
 [0.08098845 0.30310617 0.4927097  0.44379656]]
Long path (z=L) :
[[0.02195427 0.        ]
 [0.59610537 0.13159573]
 [0.         0.03362698]
 [0.00386877 0.07499637]
 [0.25995843 0.24828572]
 [0.01335859 0.51149521]
 [0.         0.        ]
 [0.10475458 0.        ]]
------------
Yielded behavior is tested [False] to being no signaling
Yielded behavior is tested [False] to being normalized


### Loop over samples to search for nontrivial alphas

In [778]:
# counter = 0
# logger.remove()

# while res.fun == 0:
#     sampled_behavior = sampler.sample()
#     srns_set = no_signaling_sets.ShortRangeNoSignalingSet(
#         delta=delta,
#         m=m,
#         measured_behavior=sampled_behavior,
#     )
#     A_eq, b_eq = srns_set.get_equations(measured_behavior=sampled_behavior)
#     q_shape = behaviors.LatentSRNSBehavior(
#         delta=delta,
#         m=m,
#     ).vector_shape[0]
#     lb = np.zeros(q_shape+1)
#     rb = np.ones(q_shape+1)
#     rb[0] = 2
#     bounds = [(lb[i], rb[i]) for i in range(q_shape+1)]
#     c = np.zeros(q_shape+1)
#     c[0] = 1

#     res:scipy.optimize.OptimizeResult = scipy.optimize.linprog(
#         c=c,
#         A_eq=A_eq,
#         b_eq=b_eq,
#         bounds=bounds,
#     )

#     counter += 1
#     if counter % 100 == 0:
#         print(f"Counter: {counter}")